# CycleGAN example for debiasing PDE solutions.

We consider the Kuramoto-Sivashinsky (KS) equation
$$\partial_t u + u\thinspace\partial_x u + \partial_{xx} u + \partial_{xxxx} u = 0,$$
a prime example of a chaotic PDE, and two datasets of its solutions computed
with [jax-cfd](https://github.com/google/jax-cfd) from the same random initial
conditions: a *reference* dataset from a pseudo-spectral solver on a fine mesh
and a *biased* dataset from a finite-volume (FV) solver on a much coarser mesh.
Because of the coarser resolution and the extra numerical dissipation of the FV
discretisation, the two datasets have different one-point statistics and energy
spectra.

The two solvers are chaotic, so their trajectories cannot be matched in time:
we only have *unpaired* samples of the two distributions. This notebook trains
a CycleGAN (Zhu et al., 2017) with two generators, $G: A \to B$ (FV to
spectral) and $F: B \to A$, and two discriminators, and then measures how much
closer the translated FV snapshots are to the spectral distribution. It uses
the same data, held-out trajectories and metrics as the rectified-flow
[notebook](https://github.com/google-research/swirl-dynamics/blob/main/swirl_dynamics/projects/debiasing/rectified_flow/colab/demo_reflow.ipynb)
and the optimal-transport
[notebook](https://github.com/google-research/swirl-dynamics/blob/main/swirl_dynamics/projects/debiasing/optimal_transport/colab/optimal_transport_sinkhorn.ipynb)
of `swirl_dynamics`, so the three approaches can be compared.

**Loss.** With $\ell$ the least-squares GAN criterion and $\lVert\cdot\rVert_1$ the
mean absolute error, the generators minimise
$$\mathcal L_{G,F} = \ell(D_B(G(a)), 1) + \ell(D_A(F(b)), 1) +
\lambda_A \lVert F(G(a)) - a\rVert_1 + \lambda_B \lVert G(F(b)) - b\rVert_1 +
\lambda_{id}\big(\lambda_B \lVert G(b) - b\rVert_1 + \lambda_A \lVert F(a) - a\rVert_1\big),$$
and each discriminator minimises
$\tfrac12[\ell(D(\text{real}), 1) + \ell(D(\text{fake}), 0)]$, where the fake
samples are drawn from a replay pool of past generator outputs. See the
package README for the full formulation and for the discussion of the network
choices (no normalisation layers and an identity output head, so that the
networks can see the amplitude of the fields).

### Downloading dependencies.

We install `swirl-dynamics` together with `flax>=0.12.8` (required for
`jax>=0.11.0`, which removed `jax.core.get_opaque_trace_state`; restart the
Colab runtime if an older `flax` was already imported).

In [ ]:
!pip install "flax>=0.12.8" git+https://github.com/google-research/swirl-dynamics.git@main

We also import all the necessary libraries.

In [ ]:
from clu import metric_writers
import h5py
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np
from orbax import checkpoint
from swirl_dynamics.projects.debiasing.cycle_gan import evaluation_metrics
from swirl_dynamics.projects.debiasing.cycle_gan import experiment
from swirl_dynamics.projects.debiasing.cycle_gan import trainers
from swirl_dynamics.projects.debiasing.cycle_gan.configs import config_ks_1d
from swirl_dynamics.templates import callbacks
from swirl_dynamics.templates import train

### Define hyper-parameters

We start from the configuration of the package (`configs/config_ks_1d.py`) and
shorten the run for the notebook. The configuration selects the 1-D CycleGAN
networks of `swirl_dynamics.lib.networks.cycle_gan` **without** normalisation
layers and with an identity output head: with instance normalisation the
discriminator is (approximately) invariant to the amplitude of its input, and
`tanh` would clip the fields to $(-1, 1)$. The loss weights
$\lambda_A = \lambda_B = 10$, $\lambda_{id} = 0.5$, the Adam optimiser
with $\beta_1 = 0.5$ and a linear learning-rate decay, and the 50-sample
replay pool follow the CycleGAN paper.

The next cell holds every knob of the notebook (the local copy of the dataset,
the working directory and the length of the run). It is tagged `parameters`, so
that tools such as [papermill](https://papermill.readthedocs.io), and the smoke
test of the package, can replace it.


In [ ]:
config = config_ks_1d.get_config()

# Local copy of the public dataset, downloaded below.
config.dataset_path = "KS_finite_volumes_vs_pseudo_spectral.hdf5"

# Shorter run for the notebook; the full config trains for 20k steps.
config.num_train_steps = 10_000
config.metric_aggregation_steps = 100
config.eval_every_steps = 500
config.num_batches_per_eval = 4
config.save_interval_steps = 1_000
config.batch_size = 32
config.eval_batch_size = 64

# Checkpoints and metrics are written here. Delete its `checkpoints` folder to
# restart the training from scratch.
workdir = "/content/cycle_gan_ks"
print(config)

### Downloading the data.

The data was generated using [jax-cfd](https://github.com/google/jax-cfd) and
uploaded to a public Google Cloud bucket. The file contains both the
finite-volume (`u_fd`) and the pseudo-spectral (`u_sp`) trajectories, the fine
grid `x` and the time stamps `t`.

We use `gsutil` to download it. In Colab it is already installed; otherwise
follow these [instructions](https://cloud.google.com/storage/docs/gsutil_install).


In [ ]:
!gsutil cp gs://gresearch/swirl_dynamics/downscaling/KS_finite_volumes_vs_pseudo_spectral.hdf5 .

We plot one trajectory of each solver. The pseudo-spectral data has four times
the spatial resolution of the finite-volume data; the dataloaders sub-sample it
by `config.downsample_factor = 4` so that both domains live on the same
48-point grid. (The periodic grid `x_` below is sub-sampled **once**, to the
finite-volume resolution, and is only used to label the axes.)


In [ ]:
with h5py.File(config.dataset_path, "r") as file:
  u_fv = file["u_fd"][()]  # Trajectories of the finite-volume solver.
  u_sp = file["u_sp"][()]  # Trajectories of the pseudo-spectral solver.
  x = file["x"][()]  # Fine grid.
  t = file["t"][()]  # Time stamps.

# Periodic grid (with its end point) at the finite-volume resolution.
x_ = np.concatenate([x, [x[-1] + x[1] - x[0]]])[:: config.downsample_factor]
print(f"Finite-volume trajectories: {u_fv.shape}")
print(f"Pseudo-spectral trajectories: {u_sp.shape}")

plot_idx = 1
for name, u in (("Finite volumes", u_fv), ("Pseudo-spectral", u_sp)):
  plt.figure(figsize=(14, 4))
  plt.imshow(u[plot_idx].T, extent=[t[0], t[-1], x_[-1], x_[0]], aspect=5)
  plt.title(name)
  plt.xlabel("time")
  plt.ylabel("x")
  plt.show()

### Defining the dataloaders.

`experiment.load_datasets` splits the file into training and held-out snapshots
(the last `config.num_test_trajectories` trajectories are held out, as in the
other two notebooks) and computes the normalisation statistics on the training
split. `experiment.build_dataloaders` wraps them in two independent
[pygrain](https://github.com/google/grain) streams, so that each batch
`{"real_data_a", "real_data_b"}` is sampled from the product
$\mu_A \otimes \mu_B$ of the two marginals; the evaluation loader reuses the
training statistics.


In [ ]:
datasets = experiment.load_datasets(config)
train_dataloader, eval_dataloader = experiment.build_dataloaders(
    datasets, config
)

print(
    f"Training snapshots: A {datasets.train_a.shape}, B"
    f" {datasets.train_b.shape}"
)
print(
    f"Held-out snapshots: A {datasets.test_a.shape}, B {datasets.test_b.shape}"
)

batch = next(train_dataloader)
print(batch.keys())
assert batch["real_data_a"].shape == (
    config.batch_size,
    config.sample_length,
    1,
)
print(batch["real_data_a"].shape, batch["real_data_b"].shape)

## Defining the model and the trainer

`experiment.build_model` instantiates the two generators and the two
discriminators from the config and wraps them in `models.CycleGANModel`;
`experiment.build_trainer` adds the two Adam optimisers with the linear decay,
the replay pool and the alternating generator / discriminator step.


In [ ]:
model = experiment.build_model(config)
trainer = experiment.build_trainer(config, model)

num_params = [
    sum(leaf.size for leaf in jax.tree_util.tree_leaves(p))
    for p in trainer.train_state.params
]
print("Parameters (gen_a2b, gen_b2a, dis_a, dis_b):", num_params)

### Running the training loop.

Each step performs one update of both generators followed by one update of
both discriminators on the same batch. The networks are small (see the
parameter counts above), so the run is short on any accelerator; the training
and evaluation metrics are logged every `config.metric_aggregation_steps`.


In [ ]:
ckpt_options = checkpoint.CheckpointManagerOptions(
    save_interval_steps=config.save_interval_steps,
    max_to_keep=config.max_checkpoints_to_keep,
)

train.run(
    train_dataloader=train_dataloader,
    trainer=trainer,
    workdir=workdir,
    total_train_steps=config.num_train_steps,
    metric_writer=metric_writers.create_default_writer(
        workdir, asynchronous=False
    ),
    metric_aggregation_steps=config.metric_aggregation_steps,
    eval_dataloader=eval_dataloader,
    eval_every_steps=config.eval_every_steps,
    num_batches_per_eval=config.num_batches_per_eval,
    callbacks=(
        # Saves model checkpoints periodically.
        callbacks.TrainStateCheckpoint(base_dir=workdir, options=ckpt_options),
        # Reports the training progress.
        callbacks.ProgressReport(num_train_steps=config.num_train_steps),
    ),
)

# Running inference.

We restore the last checkpoint and build the deterministic generator maps
$G$ (FV to spectral) and $F$ (spectral to FV). The generators act on
normalised data, so the translated samples are mapped back to physical units
with the statistics of the training split of domain B.


In [ ]:
trained_state = trainers.CycleGANTrainState.restore_from_orbax_ckpt(
    f"{workdir}/checkpoints"
)
use_ema = config.ema_decay is not None
to_spectral = jax.jit(
    trainers.CycleGANTrainer.inference_fn_from_state_dict(
        trained_state, model, direction="a2b", use_ema=use_ema
    )
)
to_fv = jax.jit(
    trainers.CycleGANTrainer.inference_fn_from_state_dict(
        trained_state, model, direction="b2a", use_ema=use_ema
    )
)

stats_a, stats_b = datasets.stats_a, datasets.stats_b
test_fv = datasets.test_a  # Held-out finite-volume snapshots (A).
test_sp = datasets.test_b  # Held-out pseudo-spectral snapshots (B).

normalised_fv = (
    test_fv if stats_a is None else (test_fv - stats_a["mean"]) / stats_a["std"]
)
translated = experiment.translate_in_batches(
    to_spectral, jnp.asarray(normalised_fv), config.eval_batch_size
)
recovered = experiment.translate_in_batches(
    to_fv, translated, config.eval_batch_size
)
debiased = np.asarray(
    translated
    if stats_b is None
    else translated * stats_b["std"] + stats_b["mean"]
)
recovered_fv = np.asarray(
    recovered
    if stats_a is None
    else recovered * stats_a["std"] + stats_a["mean"]
)
print(f"Translated snapshots: {debiased.shape}")

### Evaluation of the transformed data.

We look at one held-out snapshot, its translation and its cycle
reconstruction. The cycle-consistency term keeps $F(G(a))$ close to $a$, so
the translation is expected to change the small scales, which carry most of
the bias, more than the large ones.


In [ ]:
idx = 0
plt.figure(figsize=(8, 3))
plt.plot(test_fv[idx, :, 0], label="Finite volumes $a$")
plt.plot(debiased[idx, :, 0], label="Translated $G(a)$")
plt.plot(recovered_fv[idx, :, 0], "--", label="Reconstructed $F(G(a))$")
plt.xlabel("grid index")
plt.legend()
plt.show()

We then compare the marginal statistics at one point of the domain (the
problem is translation equivariant, so one point is representative), as in the
rectified-flow and optimal-transport notebooks.


In [ ]:
idx_x = 2
plt.figure()
plt.hist(
    test_fv[:, idx_x, 0],
    bins=50,
    density=True,
    alpha=0.5,
    label="Finite volumes",
)
plt.hist(
    test_sp[:, idx_x, 0], bins=50, density=True, alpha=0.5, label="Spectral"
)
plt.hist(
    debiased[:, idx_x, 0], bins=50, density=True, alpha=0.5, label="CycleGAN"
)
plt.legend()
plt.show()

Then we quantify the debiasing with the metrics of
`evaluation_metrics`: the $L^1$ distance between the normalised histograms of
all values, the one-dimensional Wasserstein-1 distance, the energy-weighted
absolute $\log_{10}$ ratio of the energy spectra, and the relative
cycle-consistency error $\lVert F(G(a)) - a\rVert_2 / \lVert a\rVert_2$. The `*_reference`
values are the same distances between the untranslated finite-volume data and
the spectral data; a successful debiasing moves the first three metrics well
below their reference values.

In [ ]:
metrics = experiment.evaluate(config, datasets, trained_state, model)
for name, value in metrics.items():
  print(f"{name:32s} {value:.4f}")

Finally we plot the mean energy spectra of the three datasets.

In [ ]:
wavenumbers = np.arange(config.sample_length // 2 + 1)
for name, samples in (
    ("Finite volumes", test_fv),
    ("Spectral", test_sp),
    ("CycleGAN", debiased),
):
  spectrum = evaluation_metrics.energy_spectrum_1d(jnp.asarray(samples))
  plt.loglog(wavenumbers[1:], spectrum[1:], label=name)
plt.xlabel("wavenumber")
plt.ylabel("energy")
plt.legend()
plt.show()

print(
    "Spectral log ratio, FV vs spectral:",
    f"{metrics['spectral_log_ratio_reference']:.4f}",
)
print(
    "Spectral log ratio, CycleGAN vs spectral:",
    f"{metrics['spectral_log_ratio']:.4f}",
)